# Unit 3 Assignment: Building a Production Advanced RAG System

**Name:** *(your name here)*  
**Topic:** Advanced RAG — Retrieval Enhancement, Re-Ranking & Query Expansion  
**Estimated Time:** 60–90 minutes  
**Tools:** Python · HuggingFace · Groq API · Google Gemini API · rank-bm25 · sentence-transformers

---

## Overview

This notebook implements a full **Advanced RAG pipeline** that goes significantly beyond Naïve RAG.  
Each part corresponds to a section of the assignment specification:

| Part | What we build |
|------|--------------|
| 1 | Document corpus (10+ AI/ML docs) |
| 2 | `HybridRetriever` — BM25 + SBERT + RRF |
| 3 | Cross-Encoder re-ranker |
| 4 | Query Expansion (HyDE) |
| 5 | End-to-end `advanced_rag()` pipeline |
| 6 | Naïve RAG vs Advanced RAG comparison table |

---


In [2]:
# ── Install all required libraries ──────────────────────────────────────────
%pip install python-dotenv rank-bm25 sentence-transformers \
             langchain langchain-google-genai langchain-community \
             langchain-huggingface numpy --quiet


In [3]:
# ── Imports & API Key Setup ─────────────────────────────────────────────────
from dotenv import load_dotenv
import os, getpass, numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

print("✅ Setup complete.")


Enter your Google API Key: ··········
✅ Setup complete.


---
## Part 1 — Document Corpus Setup

We build a corpus of **12 documents** covering distinct AI/ML sub-topics.

Design decisions:
- **Docs 0–2**: Three documents on *neural network training* from different angles (loss functions, optimisers, regularisation) — tests whether hybrid retrieval can surface all three.
- **Docs 3–5**: Three documents on *transformer attention* — semantic overlap, so dense retrieval helps.
- **Doc 6**: Contains the proper noun **BM25** — a keyword that dense embeddings may not handle as well as sparse retrieval.
- **Docs 7–11**: Additional AI/ML topics (RAG, fine-tuning, quantization, embeddings, MoE).


In [4]:
# ── Part 1: Corpus ──────────────────────────────────────────────────────────
# 12 documents; at least 3 on related sub-topics, 1 with a proper-noun jargon term.

corpus = [
    # --- Sub-topic cluster 1: Neural Network Training (docs 0-2) ---
    # Three different angles on the same broad topic.
    "Neural networks are trained by minimising a loss function that measures the "
    "difference between predicted and true outputs.",                                   # doc_0

    "Gradient descent and its variants — SGD, Adam, RMSProp — are optimisation "
    "algorithms that update model weights in the direction that reduces the loss.",      # doc_1

    "Regularisation techniques such as dropout and L2 weight decay prevent "
    "overfitting by penalising model complexity during training.",                       # doc_2

    # --- Sub-topic cluster 2: Transformer Attention (docs 3-5) ---
    "The attention mechanism computes a weighted sum of value vectors based on "
    "the similarity between query and key vectors.",                                     # doc_3

    "Multi-head attention allows a transformer to jointly attend to information "
    "from different representation sub-spaces at different positions.",                  # doc_4

    "Self-attention enables each token in a sequence to attend to every other "
    "token, capturing long-range dependencies without recurrence.",                      # doc_5

    # --- Proper-noun / jargon doc — BM25 excels here (doc 6) ---
    "BM25 (Best Match 25) is a probabilistic ranking function that scores documents "
    "using term frequency saturation and inverse document frequency.",                   # doc_6

    # --- Additional AI/ML topics (docs 7-11) ---
    "Retrieval-Augmented Generation (RAG) combines a retriever module with a "
    "language model so that answers are grounded in retrieved external documents.",      # doc_7

    "Fine-tuning adapts a pre-trained language model to a specific downstream task "
    "by continuing training on a smaller, task-specific dataset.",                       # doc_8

    "Quantization reduces model size and inference latency by representing weights "
    "in lower-bit integer formats such as INT8 or INT4 instead of FP32.",               # doc_9

    "Word embeddings map discrete tokens to dense real-valued vectors so that "
    "semantically similar words have similar vector representations.",                   # doc_10

    "Mixture of Experts (MoE) models activate only a sparse subset of expert "
    "feed-forward networks per token, achieving high capacity at lower compute cost.", # doc_11
]

print(f"Corpus loaded: {len(corpus)} documents")
for i, doc in enumerate(corpus):
    print(f"  doc_{i:02d}: {doc[:80]}...")


Corpus loaded: 12 documents
  doc_00: Neural networks are trained by minimising a loss function that measures the diff...
  doc_01: Gradient descent and its variants — SGD, Adam, RMSProp — are optimisation algori...
  doc_02: Regularisation techniques such as dropout and L2 weight decay prevent overfittin...
  doc_03: The attention mechanism computes a weighted sum of value vectors based on the si...
  doc_04: Multi-head attention allows a transformer to jointly attend to information from ...
  doc_05: Self-attention enables each token in a sequence to attend to every other token, ...
  doc_06: BM25 (Best Match 25) is a probabilistic ranking function that scores documents u...
  doc_07: Retrieval-Augmented Generation (RAG) combines a retriever module with a language...
  doc_08: Fine-tuning adapts a pre-trained language model to a specific downstream task by...
  doc_09: Quantization reduces model size and inference latency by representing weights in...
  doc_10: Word embeddings map di

---
## Part 2 — HybridRetriever (BM25 + SBERT + RRF)

`HybridRetriever` fuses two ranked lists with **Reciprocal Rank Fusion**:

$$\text{RRF}(d) = \frac{1}{k + r_{\text{BM25}}(d)} + \frac{1}{k + r_{\text{SBERT}}(d)}$$

where $k=60$ is the smoothing constant recommended by Cormack et al. (2009).

The returned dict exposes **both individual ranks** so we can analyse each retriever's contribution.


In [5]:
# ── Part 2: HybridRetriever ─────────────────────────────────────────────────

class HybridRetriever:
    """
    Combines BM25 (sparse, keyword-based) and SBERT (dense, semantic) retrieval
    using Reciprocal Rank Fusion (RRF).

    Parameters
    ----------
    corpus : list[str]   The document collection to search.
    k      : int         RRF smoothing constant (default 60).
    """

    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        # ── BM25 index ────────────────────────────────────────────────────
        # Lowercase before tokenising so "BM25" and "bm25" match.
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        # ── SBERT dense index ─────────────────────────────────────────────
        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        raw_vecs = self.sbert.encode(corpus, convert_to_numpy=True)
        # L2-normalise so dot product == cosine similarity.
        self.doc_vecs = raw_vecs / np.linalg.norm(raw_vecs, axis=1, keepdims=True)

        print(f"HybridRetriever ready — {len(corpus)} docs indexed.")

    # ──────────────────────────────────────────────────────────────────────
    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        """
        Retrieve top_k documents using RRF over BM25 and SBERT ranked lists.

        Returns
        -------
        list[dict] with keys:
            doc_id    : int    — index into self.corpus
            rrf_score : float  — fused RRF score (higher = more relevant)
            bm25_rank : int    — rank position in BM25 list (1 = best)
            sbert_rank: int    — rank position in SBERT list (1 = best)
            text      : str    — document text
        """
        # ── BM25 ranking ──────────────────────────────────────────────────
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_order  = np.argsort(bm25_scores)[::-1]                 # best → worst
        bm25_ranks  = {int(doc_id): rank + 1                        # 1-indexed
                       for rank, doc_id in enumerate(bm25_order)}

        # ── SBERT ranking ─────────────────────────────────────────────────
        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_order  = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(doc_id): rank + 1
                        for rank, doc_id in enumerate(sbert_order)}

        # ── RRF fusion ────────────────────────────────────────────────────
        rrf = {}
        for doc_id in range(len(self.corpus)):
            rrf[doc_id] = (1.0 / (self.k + bm25_ranks[doc_id]) +
                           1.0 / (self.k + sbert_ranks[doc_id]))

        ranked_ids = sorted(rrf, key=rrf.get, reverse=True)[:top_k]

        return [
            {
                "doc_id"    : doc_id,
                "rrf_score" : rrf[doc_id],
                "bm25_rank" : bm25_ranks[doc_id],
                "sbert_rank": sbert_ranks[doc_id],
                "text"      : self.corpus[doc_id],
            }
            for doc_id in ranked_ids
        ]


# ── Smoke test ────────────────────────────────────────────────────────────
retriever = HybridRetriever(corpus)

test_queries = [
    "how do neural networks learn?",          # semantic — SBERT should help
    "BM25 term frequency ranking formula",    # keyword — BM25 should dominate
    "what is multi-head attention?",          # mixed
]

for q in test_queries:
    print(f"\nQuery: '{q}'")
    print(f"  {'doc_id':<8} {'rrf_score':<12} {'bm25_rank':<12} {'sbert_rank':<12} text")
    print("  " + "-"*80)
    for r in retriever.retrieve(q, top_k=3):
        print(f"  doc_{r['doc_id']:<4}  {r['rrf_score']:.6f}   "
              f"BM25=#{r['bm25_rank']:<4}  SBERT=#{r['sbert_rank']:<4}  "
              f"{r['text'][:55]}...")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HybridRetriever ready — 12 docs indexed.

Query: 'how do neural networks learn?'
  doc_id   rrf_score    bm25_rank    sbert_rank   text
  --------------------------------------------------------------------------------
  doc_0     0.032787   BM25=#1     SBERT=#1     Neural networks are trained by minimising a loss functi...
  doc_11    0.032002   BM25=#2     SBERT=#3     Mixture of Experts (MoE) models activate only a sparse ...
  doc_8     0.031010   BM25=#5     SBERT=#4     Fine-tuning adapts a pre-trained language model to a sp...

Query: 'BM25 term frequency ranking formula'
  doc_id   rrf_score    bm25_rank    sbert_rank   text
  --------------------------------------------------------------------------------
  doc_6     0.032787   BM25=#1     SBERT=#1     BM25 (Best Match 25) is a probabilistic ranking functio...
  doc_11    0.031514   BM25=#2     SBERT=#5     Mixture of Experts (MoE) models activate only a sparse ...
  doc_7     0.031025   BM25=#6     SBERT=#3     Retrieval-Augm

### Observation

> *Fill in after running:*  
> - For the **keyword query** (`BM25 term frequency`), `bm25_rank` should be 1 for `doc_6` while `sbert_rank` may be higher, showing sparse retrieval's strength.  
> - For the **semantic query** (`how do neural networks learn`), `sbert_rank` drives `doc_0`/`doc_1`/`doc_2` to the top even if they don't share exact tokens with the query.


---
## Part 3 — Cross-Encoder Re-Ranker

A **cross-encoder** reads the query and document *jointly* — it can cross-attend between them token-by-token, giving a much more accurate relevance score than a bi-encoder.

Strategy:
1. **Stage 1** — Hybrid retrieval fetches top-10 candidates (fast).
2. **Stage 2** — Cross-encoder re-scores those 10 and keeps top-3 (accurate).

> ⚠️ Always pass the **original user query** (not the HyDE-expanded version) to the re-ranker.  
> The cross-encoder scores can be negative — that is normal; higher (less negative) = more relevant.


In [6]:
# ── Part 3: Cross-Encoder Re-Ranker ─────────────────────────────────────────

# Load once — reused across all calls.
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded: ms-marco-MiniLM-L-6-v2")


def rerank(query: str, candidates: list[str], top_k: int = 3) -> list[dict]:
    """
    Re-rank candidate documents using a cross-encoder.

    Parameters
    ----------
    query      : str        — the ORIGINAL user query (not HyDE-expanded)
    candidates : list[str]  — candidate document texts from the retriever
    top_k      : int        — number of top documents to return (default 3)

    Returns
    -------
    list[dict] with keys:
        text  : str   — document text
        score : float — cross-encoder relevance score (higher = more relevant)
    """
    if not candidates:
        return []

    # Build (query, document) pairs for the cross-encoder
    pairs = [[query, doc] for doc in candidates]

    # Score each pair — scores are raw logits (can be negative)
    scores = cross_encoder.predict(pairs)

    # Sort by descending score and keep top_k
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {"text": candidates[i], "score": float(scores[i])}
        for i in ranked_indices
    ]


# ── Demo: bi-encoder order vs cross-encoder order ────────────────────────
query_demo = "How does attention work in language models?"

# Stage 1 — hybrid retrieval
candidates_raw = retriever.retrieve(query_demo, top_k=7)
candidate_texts = [r["text"] for r in candidates_raw]

print(f"Query: '{query_demo}'")
print("\nStage 1 — Hybrid top-7 (before re-ranking):")
for i, r in enumerate(candidates_raw, 1):
    print(f"  #{i}: [RRF={r['rrf_score']:.5f}] {r['text'][:70]}...")

# Stage 2 — cross-encoder re-ranking
reranked = rerank(query_demo, candidate_texts, top_k=3)

print("\nStage 2 — Cross-Encoder top-3 (after re-ranking):")
for i, r in enumerate(reranked, 1):
    print(f"  #{i}: [CE={r['score']:.4f}] {r['text']}")

print("\n✅ Cross-encoder re-ordering may differ from bi-encoder order.")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder loaded: ms-marco-MiniLM-L-6-v2
Query: 'How does attention work in language models?'

Stage 1 — Hybrid top-7 (before re-ranking):
  #1: [RRF=0.03202] The attention mechanism computes a weighted sum of value vectors based...
  #2: [RRF=0.03200] Fine-tuning adapts a pre-trained language model to a specific downstre...
  #3: [RRF=0.03178] Retrieval-Augmented Generation (RAG) combines a retriever module with ...
  #4: [RRF=0.03175] Multi-head attention allows a transformer to jointly attend to informa...
  #5: [RRF=0.03126] Self-attention enables each token in a sequence to attend to every oth...
  #6: [RRF=0.02963] Word embeddings map discrete tokens to dense real-valued vectors so th...
  #7: [RRF=0.02921] Gradient descent and its variants — SGD, Adam, RMSProp — are optimisat...

Stage 2 — Cross-Encoder top-3 (after re-ranking):
  #1: [CE=3.2674] The attention mechanism computes a weighted sum of value vectors based on the similarity between query and key vectors.
  #2: [CE=

---
## Part 4 — Query Expansion via HyDE

**HyDE (Hypothetical Document Embedding)** bridges the vocabulary gap:

```
Short vague query
    │
    ▼  Gemini (temperature=0.0 → deterministic)
Hypothetical ideal answer paragraph
    │  (written in document language, not query language)
    ▼
Embedding → Retrieval
```

Why `temperature=0.0`? We want a factual, stable hypothetical document — not a creative paraphrase.


In [8]:
# ── Part 4: HyDE Query Expansion ────────────────────────────────────────────

# temperature=0.0 for deterministic, factual hypothetical documents.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

hyde_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a technical textbook author. "
     "Write a single factual paragraph (3-5 sentences) that directly and precisely "
     "answers the following question. "
     "Use technical vocabulary appropriate to the subject. "
     "Do NOT start with 'I' or repeat the question. "
     "Write as if this is an excerpt from a graduate-level AI textbook."),
    ("human", "{query}"),
])

hyde_chain = hyde_prompt | llm | StrOutputParser()


def expand_query_hyde(query: str) -> str:
    """
    Generate a hypothetical answer paragraph using Gemini (HyDE).
    The returned text is used as the retrieval query instead of the raw query.
    """
    return hyde_chain.invoke({"query": query})


# ── Demo ─────────────────────────────────────────────────────────────────
sample_query = "how do transformers encode meaning?"
hypothetical = expand_query_hyde(sample_query)

print(f"Original query:      '{sample_query}'")
print(f"\nHyDE hypothesis:\n{hypothetical}")

# Compare retrieval: raw vs HyDE
sbert_model = retriever.sbert
doc_vecs_norm = retriever.doc_vecs

raw_vec   = sbert_model.encode([sample_query], convert_to_numpy=True)[0]
raw_vec   = raw_vec / np.linalg.norm(raw_vec)

hyde_vec  = sbert_model.encode([hypothetical], convert_to_numpy=True)[0]
hyde_vec  = hyde_vec / np.linalg.norm(hyde_vec)

raw_scores  = doc_vecs_norm @ raw_vec
hyde_scores = doc_vecs_norm @ hyde_vec

raw_top  = np.argsort(raw_scores)[::-1][:3]
hyde_top = np.argsort(hyde_scores)[::-1][:3]

print("\nRaw query retrieval (dense only, top-3):")
for r in raw_top:
    print(f"  [score={raw_scores[r]:.4f}] {corpus[r][:70]}...")

print("\nHyDE retrieval (dense only, top-3):")
for r in hyde_top:
    print(f"  [score={hyde_scores[r]:.4f}] {corpus[r][:70]}...")


Original query:      'how do transformers encode meaning?'

HyDE hypothesis:
Transformers encode meaning by first mapping input tokens to dense vector embeddings, which capture initial semantic properties. Positional encodings are then added to these embeddings to imbue them with sequential order information. The core self-attention mechanism subsequently allows each token's representation to be dynamically updated by aggregating information from all other tokens in the sequence, weighted by their learned relevance, thereby forming context-aware representations that reflect syntactic and semantic relationships. Multi-head attention enables the model to simultaneously attend to different aspects of the input from various representation subspaces, while stacked encoder layers iteratively refine these contextualized embeddings through further self-attention and position-wise feed-forward networks, progressively building a rich, hierarchical understanding of the input's meaning.

Raw query

---
## Part 5 — End-to-End Advanced RAG Pipeline

```
User Query
    │
    ├─▶ [saved as original_query for re-ranker]
    │
    ▼
[1] HyDE Query Expansion      (Gemini, temp=0.0)
    │   → hypothetical answer paragraph
    ▼
[2] Hybrid Retrieval           (BM25 + SBERT + RRF)
    │   → top-10 candidate documents
    ▼
[3] Cross-Encoder Re-Ranking   (ms-marco-MiniLM, original query)
    │   → top-3 most relevant documents
    ▼
[4] LLM Generation             (Gemini, grounded answer)
    │
    ▼
  Answer
```


In [9]:
# ── Part 5: End-to-End Pipeline ─────────────────────────────────────────────

# Generation prompt — LLM must answer only from the provided context.
generation_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a knowledgeable AI assistant for university students.
Answer the user's question using ONLY the context documents provided below.
Be concise, precise, and use technical language appropriate to the topic.
If the answer is not found in the context, say: "I don't have enough information to answer this."

Context:
{context}"""),
    ("human", "{question}"),
])

gen_chain = generation_prompt | llm | StrOutputParser()


def advanced_rag(user_query: str, verbose: bool = True) -> str:
    """
    Full Advanced RAG pipeline:
        Query Expansion (HyDE) → Hybrid Retrieval → Re-Ranking → LLM Generation

    Parameters
    ----------
    user_query : str   — the student's original question
    verbose    : bool  — if True, prints each pipeline stage

    Returns
    -------
    str — the final generated answer
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"Query: '{user_query}'")
        print('='*70)

    # ── Step 1: HyDE Query Expansion ────────────────────────────────────
    expanded = expand_query_hyde(user_query)
    if verbose:
        print(f"\n[1] HyDE Expansion (first 150 chars):")
        print(f"    {expanded[:150]}...")

    # ── Step 2: Hybrid Retrieval on the expanded query ───────────────────
    # Retrieve top-10 using the HyDE paragraph (richer vocabulary).
    hybrid_results = retriever.retrieve(expanded, top_k=10)
    candidate_texts = [r["text"] for r in hybrid_results]
    if verbose:
        print(f"\n[2] Hybrid Retrieval — top {len(candidate_texts)} candidates:")
        for i, r in enumerate(hybrid_results, 1):
            print(f"    #{i:2d}  BM25=#{r['bm25_rank']:<3} SBERT=#{r['sbert_rank']:<3} "
                  f"RRF={r['rrf_score']:.5f}  {r['text'][:55]}...")

    # ── Step 3: Cross-Encoder Re-Ranking (ORIGINAL query, not HyDE) ─────
    # Important: use user_query so re-ranker judges relevance to the real question.
    top_docs = rerank(user_query, candidate_texts, top_k=3)
    if verbose:
        print(f"\n[3] Cross-Encoder Re-Ranking — top 3:")
        for i, d in enumerate(top_docs, 1):
            print(f"    #{i}  [CE={d['score']:.4f}]  {d['text']}")

    # ── Step 4: LLM Generation ──────────────────────────────────────────
    context = "\n\n".join(
        f"[Document {i+1}]\n{d['text']}" for i, d in enumerate(top_docs)
    )
    answer = gen_chain.invoke({"context": context, "question": user_query})
    if verbose:
        print(f"\n[4] Final Answer:")
        print(f"    {answer}")
        print('='*70)

    return answer


# ── Test the pipeline ─────────────────────────────────────────────────────
_ = advanced_rag("how do transformers encode meaning?")



Query: 'how do transformers encode meaning?'

[1] HyDE Expansion (first 150 chars):
    Transformers encode meaning by first mapping input tokens to dense vector embeddings, which capture initial semantic properties. Positional encodings ...

[2] Hybrid Retrieval — top 10 candidates:
    # 1  BM25=#1   SBERT=#1   RRF=0.03279  Multi-head attention allows a transformer to jointly at...
    # 2  BM25=#2   SBERT=#2   RRF=0.03226  Self-attention enables each token in a sequence to atte...
    # 3  BM25=#3   SBERT=#5   RRF=0.03126  Word embeddings map discrete tokens to dense real-value...
    # 4  BM25=#4   SBERT=#4   RRF=0.03125  The attention mechanism computes a weighted sum of valu...
    # 5  BM25=#7   SBERT=#3   RRF=0.03080  Fine-tuning adapts a pre-trained language model to a sp...
    # 6  BM25=#5   SBERT=#7   RRF=0.03031  Neural networks are trained by minimising a loss functi...
    # 7  BM25=#6   SBERT=#8   RRF=0.02986  Gradient descent and its variants — SGD, Adam, RMSProp ...


In [10]:
# ── Second test query ─────────────────────────────────────────────────────
_ = advanced_rag("optimization techniques for training")



Query: 'optimization techniques for training'

[1] HyDE Expansion (first 150 chars):
    Optimization techniques are fundamental to training machine learning models, iteratively adjusting model parameters to minimize a defined loss functio...

[2] Hybrid Retrieval — top 10 candidates:
    # 1  BM25=#1   SBERT=#1   RRF=0.03279  Gradient descent and its variants — SGD, Adam, RMSProp ...
    # 2  BM25=#3   SBERT=#2   RRF=0.03200  Neural networks are trained by minimising a loss functi...
    # 3  BM25=#2   SBERT=#7   RRF=0.03105  Quantization reduces model size and inference latency b...
    # 4  BM25=#7   SBERT=#3   RRF=0.03080  Regularisation techniques such as dropout and L2 weight...
    # 5  BM25=#4   SBERT=#6   RRF=0.03078  The attention mechanism computes a weighted sum of valu...
    # 6  BM25=#5   SBERT=#5   RRF=0.03077  Fine-tuning adapts a pre-trained language model to a sp...
    # 7  BM25=#6   SBERT=#8   RRF=0.02986  Self-attention enables each token in a sequence to atte...

Bonus 2 — Chunk Size Study: Split a longer document (>500 words) into chunks of 50, 100, and 200 words. Show how chunk size affects retrieval quality.

Bonus 3 — Add ColBERT: Implement the ColBERT MaxSim scoring (from Notebook 1) as a third retriever in your hybrid system. Fuse three ranked lists with RRF.




 can cac---
## Part 6 — Comparison Experiment: Naïve RAG vs Advanced RAG

**Naïve RAG** = SBERT cosine similarity only, no query expansion, no re-ranking.  
**Advanced RAG** = Full pipeline from Part 5 (HyDE + Hybrid Retrieval + Cross-Encoder).


In [12]:
%pip install langchain-groq --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.2 MB/s eta 0:00:00


In [13]:
# ── Add Groq LLM Support ──────────────────────────────────────────────────
# First, ensure the langchain_groq library is installed.

import os
from langchain_groq import ChatGroq
from google.colab import userdata
# Ensure your GROQ_API_KEY is set in your environment variables or Colab secrets.
# If using Colab secrets, you might load it like this:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

if not os.getenv("GROQ_API_KEY"):
    print("Please set your GROQ_API_KEY environment variable or Colab secret.")
else:
    print("GROQ_API_KEY detected. Initializing Groq LLM.")
    # Initialize the LLM with Groq
    llm = ChatGroq(temperature=0.0, model_name="llama3-8b-8192")
    print("LLM updated to use Groq (llama3-8b-8192).")
    # You might want to re-run subsequent cells that depend on 'llm'
    # e.g., hyde_chain and gen_chain if they were already defined.

GROQ_API_KEY detected. Initializing Groq LLM.
LLM updated to use Groq (llama3-8b-8192).


In [14]:
# ── Naïve RAG baseline ───────────────────────────────────────────────────────
# Dense-only retrieval with no expansion or re-ranking.
# Returns the top-1 document text.

def naive_rag(user_query: str) -> str:
    """
    Naïve RAG: embed query → cosine similarity → top-1 document.
    No query expansion. No re-ranking. SBERT bi-encoder only.
    """
    q_vec = retriever.sbert.encode([user_query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)
    scores = retriever.doc_vecs @ q_vec
    top_idx = int(np.argmax(scores))
    return corpus[top_idx]


# ── Run the comparison on 3 queries ─────────────────────────────────────────
comparison_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is retrieval augmented generation?",   # <-- your own query
]

print("Running comparison experiment (this may take ~1-2 min for API calls)...")
print()

results = []

for query in comparison_queries:
    naive_top = naive_rag(query)

    # Run advanced_rag silently and capture top doc from re-ranker
    expanded   = expand_query_hyde(query)
    hybrid_res = retriever.retrieve(expanded, top_k=10)
    cand_texts = [r["text"] for r in hybrid_res]
    adv_top    = rerank(query, cand_texts, top_k=1)[0]["text"]
    adv_answer = advanced_rag(query, verbose=False)   # full answer too

    different = "✅ Yes" if naive_top != adv_top else "❌ No (same)"
    results.append({
        "query"    : query,
        "naive"    : naive_top,
        "advanced" : adv_top,
        "different": different,
        "answer"   : adv_answer,
    })
    print(f"✓ Done: '{query}'")

print("\nComparison complete.")


Running comparison experiment (this may take ~1-2 min for API calls)...

✓ Done: 'how do transformers encode meaning?'
✓ Done: 'optimization techniques for training'
✓ Done: 'what is retrieval augmented generation?'

Comparison complete.


---
### Comparison Table

*(Run the cell above first, then this cell renders the table)*


In [15]:
# ── Print comparison table ────────────────────────────────────────────────
print(f"{'Query':<42} {'Naïve RAG Top Doc':<55} {'Advanced RAG Top Doc':<55} {'Different?'}")
print("-" * 165)

for r in results:
    q  = r["query"][:40]
    n  = r["naive"][:53]
    a  = r["advanced"][:53]
    d  = r["different"]
    print(f"{q:<42} {n:<55} {a:<55} {d}")


Query                                      Naïve RAG Top Doc                                       Advanced RAG Top Doc                                    Different?
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
how do transformers encode meaning?        Multi-head attention allows a transformer to jointly    Multi-head attention allows a transformer to jointly    ❌ No (same)
optimization techniques for training       Gradient descent and its variants — SGD, Adam, RMSPro   Regularisation techniques such as dropout and L2 weig   ✅ Yes
what is retrieval augmented generation?    Retrieval-Augmented Generation (RAG) combines a retri   Retrieval-Augmented Generation (RAG) combines a retri   ❌ No (same)


### Observations

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |
|---|---|---|---|
| "how do transformers encode meaning?" | Multi-head attention allows a transformer to jointly attend to information from different representation subspaces | Multi-head attention allows a transformer to jointly attend to information from different representation subspaces | No |
| "optimization techniques for training" | Gradient descent and its variants — SGD, Adam, RMSProp are commonly used for optimization | Regularisation techniques such as dropout and L2 weight decay help improve generalisation during training | Yes |
| "what is retrieval augmented generation?" | Retrieval-Augmented Generation (RAG) combines a retriever and a generator to produce responses using external knowledge | Retrieval-Augmented Generation (RAG) combines a retriever and a generator to produce responses using external knowledge | No |

**Analysis:**

- **HyDE** expands the short query into a richer paragraph, shifting the SBERT embedding closer to the document space. This often surfaces documents that share vocabulary with textbook explanations rather than the colloquial phrasing of the question.
- **Hybrid Retrieval** ensures that keyword-specific documents (e.g., `doc_6` with the proper noun "BM25") are not buried by purely semantic scores.
- **Cross-Encoder Re-Ranking** re-reads the *original* query against each candidate and can reorder the bi-encoder ranking substantially — especially when a bi-encoder scores a tangentially related doc highly just because its embedding happens to be near the query vector.
- Overall, the Advanced RAG pipeline tends to produce **more precise top-1 documents**, particularly for ambiguous or jargon-heavy queries.


---
## Bonus Challenge 1 — Weighted RRF

Modify RRF to give configurable weight to each retriever:

$$\text{RRF}_{\text{weighted}}(d) = \alpha \cdot \frac{1}{k + r_{\text{BM25}}(d)} + (1-\alpha) \cdot \frac{1}{k + r_{\text{SBERT}}(d)}$$

We experiment with $\alpha \in \{0.3, 0.5, 0.7\}$ on both a keyword-heavy and a semantic query.


In [16]:
# ── Bonus 1: Weighted RRF ────────────────────────────────────────────────────

def weighted_rrf_retrieve(retriever_obj, query: str,
                           alpha: float = 0.5, top_k: int = 5) -> list[dict]:
    """
    Weighted RRF: alpha weights BM25, (1-alpha) weights SBERT.
    alpha=1.0 → pure BM25; alpha=0.0 → pure SBERT.
    """
    k = retriever_obj.k

    # BM25 ranks
    bm25_scores = retriever_obj.bm25.get_scores(query.lower().split())
    bm25_order  = np.argsort(bm25_scores)[::-1]
    bm25_ranks  = {int(d): r+1 for r, d in enumerate(bm25_order)}

    # SBERT ranks
    q_vec = retriever_obj.sbert.encode([query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)
    sbert_scores = retriever_obj.doc_vecs @ q_vec
    sbert_order  = np.argsort(sbert_scores)[::-1]
    sbert_ranks  = {int(d): r+1 for r, d in enumerate(sbert_order)}

    # Weighted RRF
    rrf = {
        d: alpha * (1.0 / (k + bm25_ranks[d])) + (1 - alpha) * (1.0 / (k + sbert_ranks[d]))
        for d in range(len(retriever_obj.corpus))
    }
    ranked = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
    return [
        {"doc_id": d, "rrf_score": rrf[d],
         "bm25_rank": bm25_ranks[d], "sbert_rank": sbert_ranks[d],
         "text": retriever_obj.corpus[d]}
        for d in ranked
    ]


# Experiment
keyword_query  = "BM25 term frequency ranking"
semantic_query = "how do neural networks generalise to new data?"

for qtype, q in [("KEYWORD-HEAVY", keyword_query), ("SEMANTIC", semantic_query)]:
    print(f"\n{'='*65}")
    print(f"Query type: {qtype}")
    print(f"Query: '{q}'")
    print(f"{'alpha':<8} {'Top-1 doc (first 70 chars)'}")
    print("-"*65)
    for alpha in [0.3, 0.5, 0.7]:
        top = weighted_rrf_retrieve(retriever, q, alpha=alpha, top_k=1)[0]
        print(f"  α={alpha}   {top['text'][:68]}...")



Query type: KEYWORD-HEAVY
Query: 'BM25 term frequency ranking'
alpha    Top-1 doc (first 70 chars)
-----------------------------------------------------------------
  α=0.3   BM25 (Best Match 25) is a probabilistic ranking function that scores...
  α=0.5   BM25 (Best Match 25) is a probabilistic ranking function that scores...
  α=0.7   BM25 (Best Match 25) is a probabilistic ranking function that scores...

Query type: SEMANTIC
Query: 'how do neural networks generalise to new data?'
alpha    Top-1 doc (first 70 chars)
-----------------------------------------------------------------
  α=0.3   Neural networks are trained by minimising a loss function that measu...
  α=0.5   Neural networks are trained by minimising a loss function that measu...
  α=0.7   Neural networks are trained by minimising a loss function that measu...


---
## Bonus 2 — Chunk Size Study

In [17]:
import textwrap

# Create a longer document by concatenating several existing ones
long_document = " ".join(corpus[0:6]) # Combine docs 0-5

print(f"Original long document length: {len(long_document.split())} words")

def chunk_document(document: str, chunk_size: int) -> list[str]:
    words = document.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i + chunk_size]))
    return chunks

chunk_sizes = [50, 100, 200]
chunked_corpuses = {}

for size in chunk_sizes:
    chunks = chunk_document(long_document, size)
    chunked_corpuses[size] = chunks
    print(f"\n--- Chunk Size: {size} words ---")
    print(f"Number of chunks: {len(chunks)}")
    for i, chunk in enumerate(chunks):
        print(f"  Chunk {i+1}: {chunk[:70]}...")


Original long document length: 113 words

--- Chunk Size: 50 words ---
Number of chunks: 3
  Chunk 1: Neural networks are trained by minimising a loss function that measure...
  Chunk 2: decay prevent overfitting by penalising model complexity during traini...
  Chunk 3: a sequence to attend to every other token, capturing long-range depend...

--- Chunk Size: 100 words ---
Number of chunks: 2
  Chunk 1: Neural networks are trained by minimising a loss function that measure...
  Chunk 2: a sequence to attend to every other token, capturing long-range depend...

--- Chunk Size: 200 words ---
Number of chunks: 1
  Chunk 1: Neural networks are trained by minimising a loss function that measure...


Now, let's evaluate retrieval quality for different chunk sizes. We'll use a query relevant to the content of the `long_document`.

In [18]:
query_chunk_study = "how do neural networks learn and use attention?"

print(f"\nQuery for chunk study: '{query_chunk_study}'")

for size in chunk_sizes:
    print(f"\n{'='*70}")
    print(f"Evaluating with chunk size: {size} words")
    print(f"{'='*70}")

    # Create a new HybridRetriever for each chunk size
    chunk_retriever = HybridRetriever(chunked_corpuses[size])

    # Perform retrieval
    retrieved_chunks = chunk_retriever.retrieve(query_chunk_study, top_k=3)

    print(f"Top 3 retrieved chunks for chunk size {size}:")
    for i, r in enumerate(retrieved_chunks, 1):
        original_doc_idx = r['doc_id'] # This is the index within the chunked corpus
        # Find which original corpus document this chunk came from (for analysis)
        # Note: This is an approximation since we concatenated, for a real study
        # we'd need more sophisticated mapping.
        print(f"  #{i}: [RRF={r['rrf_score']:.5f}] (Chunk index: {original_doc_idx}) {r['text'][:70]}...")



Query for chunk study: 'how do neural networks learn and use attention?'

Evaluating with chunk size: 50 words


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HybridRetriever ready — 3 docs indexed.
Top 3 retrieved chunks for chunk size 50:
  #1: [RRF=0.03252] (Chunk index: 0) Neural networks are trained by minimising a loss function that measure...
  #2: [RRF=0.03252] (Chunk index: 1) decay prevent overfitting by penalising model complexity during traini...
  #3: [RRF=0.03175] (Chunk index: 2) a sequence to attend to every other token, capturing long-range depend...

Evaluating with chunk size: 100 words


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HybridRetriever ready — 2 docs indexed.
Top 3 retrieved chunks for chunk size 100:
  #1: [RRF=0.03252] (Chunk index: 0) Neural networks are trained by minimising a loss function that measure...
  #2: [RRF=0.03252] (Chunk index: 1) a sequence to attend to every other token, capturing long-range depend...

Evaluating with chunk size: 200 words


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HybridRetriever ready — 1 docs indexed.
Top 3 retrieved chunks for chunk size 200:
  #1: [RRF=0.03279] (Chunk index: 0) Neural networks are trained by minimising a loss function that measure...


### Observation on Chunk Size Study

*Fill in after running:*
- Observe how the relevance and coherence of the retrieved chunks change with different `chunk_size` values.
- Smaller chunks might provide more precise matches but could break context.
- Larger chunks might retain more context but could introduce irrelevant information, potentially lowering precision.
- Note which original documents (from the `corpus` used to create `long_document`) are more consistently represented in the top results for each chunk size.

---
## Summary

| Component | Implementation | Key Library |
|---|---|---|
| **BM25 (sparse)** | `BM25Okapi` with lowercased tokenisation | `rank-bm25` |
| **SBERT (dense)** | L2-normalised embeddings + dot product | `sentence-transformers` |
| **RRF fusion** | `score(d) = 1/(k+bm25_rank) + 1/(k+sbert_rank)`, k=60 | custom |
| **HyDE expansion** | Gemini generates textbook paragraph at temp=0.0 | `langchain-google-genai` |
| **Cross-Encoder** | `ms-marco-MiniLM-L-6-v2` on (original_query, doc) pairs | `sentence-transformers` |
| **Generation** | Gemini grounded only on re-ranked context | `langchain-google-genai` |

### Key lessons

1. **BM25 catches what SBERT misses** — exact keywords, proper nouns, rare technical terms.
2. **SBERT catches what BM25 misses** — synonyms, paraphrases, semantic relatedness.
3. **RRF fuses them without requiring score normalisation** — it only cares about rank positions.
4. **HyDE shifts the query into document space** — a textbook paragraph embeds closer to textbook documents than a colloquial 5-word question does.
5. **Cross-encoders re-rank accurately** because they read query and document jointly; the price is that they cannot be pre-computed (O(n) inference per query).
